# Google ADK Live + LangSmith

This notebook keeps the LangSmith Google ADK Live plugin visible and reuses the existing ADK voice loop from `voice_demo.adk.agent`.

## 1. Agent Setup

The reused ADK agent uses Gemini Live with two tools: current time and weather.

In [ ]:
import os
import uuid

from dotenv import load_dotenv

from voice_demo.adk.agent import MODEL, RECV_SAMPLE_RATE, get_time, get_weather, run
from voice_demo.workshop import console_io

load_dotenv()

PROJECT = "voice-workshop-google-adk-live"

assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY before running this notebook."
assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY before running this notebook."

[get_time.__name__, get_weather.__name__], MODEL

## 2. Tracing Setup

`LangSmithGoogleADKLivePlugin` traces the ADK Live event stream while the app loop only handles audio playback and UI.

In [ ]:
from langsmith.integrations.google_adk_live import LangSmithGoogleADKLivePlugin

thread_id = str(uuid.uuid4())
tracing_plugin = LangSmithGoogleADKLivePlugin(
    sample_rate=RECV_SAMPLE_RATE,
    thread_id_provider=lambda: thread_id,
    project_name=PROJECT,
    tags=["workshop", "google-adk-live"],
    metadata={"model": MODEL},
)

thread_id

## 3. Run

This uses the repo's console mic/speaker transport. Stop/cancel the cell to end the voice session.

In [ ]:
audio_in, audio_out, ui = console_io(RECV_SAMPLE_RATE)

await run(
    PROJECT,
    audio_in=audio_in,
    audio_out=audio_out,
    ui=ui,
    tracing_plugin=tracing_plugin,
    thread_id=thread_id,
)